# Federated Learning vs Centralized Training: Epileptic Seizure Detection
**Autor:** Ivan Betriu  
**TFM — Máster en Data Science (La Salle)**

Este notebook compara entrenamiento centralizado vs federado (FedAvg) para detección binaria de crisis epilépticas,
utilizando dos arquitecturas (MLP y SVM lineal) bajo distribuciones IID y No-IID (Dirichlet α=0.5).
Cada configuración ejecuta 5 simulaciones con semillas 42–46 para robustez estadística.

## 1. Librerías

In [ ]:
import os
import json
import time
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import flwr as fl
from flwr.common import parameters_to_ndarrays, FitIns
from flwr.server.strategy import FedAvg
from flwr.server.server import ServerConfig
import ray

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, log_loss,
    confusion_matrix, classification_report,
)
from sklearn.linear_model import SGDClassifier as SklearnSGDClassifier
from sklearn.calibration import CalibratedClassifierCV


## 2. Configuración global

In [ ]:
# ── Hiperparámetros comunes ──────────────────────────────────
CONFIG = {
    "data_path": "Epileptic Seizure Recognition.csv",
    "num_simulations": 5,
    "initial_seed": 42,
    "test_size": 0.15,
    "val_ratio": 0.15 / 0.85,
    "num_clients": 10,
    "num_rounds": 200,
    "local_epochs": 5,
    "batch_size_mlp": 32,
    "batch_size_svm": 16,

    # MLP
    "mlp_hidden": 4,
    "mlp_lr": 0.01,
    "mlp_patience_central": 100,
    "mlp_patience_federated": 20,
    "mlp_max_epochs": 1000,

    # SVM
    "svm_lr": 0.01,
    "svm_weight_decay": 0.0005,
    "svm_patience_central": 100,
    "svm_patience_federated": 20,
    "svm_max_epochs": 1000,

    # Dirichlet (Non-IID)
    "dirichlet_alpha": 0.5,
}


## 3. Definición de modelos

In [ ]:
class MLP(nn.Module):
    """Perceptrón multicapa con una capa oculta para clasificación binaria."""

    def __init__(self, in_dim: int, hidden: int = 4, out_dim: int = 1):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden, out_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.relu(self.fc1(x)))


class LinearSVM(nn.Module):
    """Modelo lineal para SVM con hinge loss."""

    def __init__(self, in_dim: int, out_dim: int = 1):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x)


def create_model(model_type: str, in_dim: int) -> nn.Module:
    """Factoría de modelos.

    Args:
        model_type: 'mlp' o 'svm'.
        in_dim: Número de features de entrada.

    Returns:
        Instancia del modelo correspondiente.
    """
    if model_type == "mlp":
        return MLP(in_dim, hidden=CONFIG["mlp_hidden"])
    elif model_type == "svm":
        return LinearSVM(in_dim)
    else:
        raise ValueError(f"model_type debe ser 'mlp' o 'svm', recibido: {model_type}")


## 4. Funciones de pérdida y utilidades

In [ ]:
def hinge_loss_fn(outputs, targets, weight_decay_lambda, model, class_weights=None):
    """Hinge loss con regularización L2 y pesos de clase opcionales.

    Args:
        outputs: Logits del modelo (N, 1).
        targets: Etiquetas binarias (N, 1), valores 0/1.
        weight_decay_lambda: Factor de regularización L2.
        model: Modelo PyTorch (para acceder a los pesos).
        class_weights: Tensor con peso para la clase positiva, o None.

    Returns:
        Pérdida escalar.
    """
    targets_transformed = targets.view_as(outputs) * 2 - 1  # {0,1} → {-1,+1}
    loss_unreduced = torch.max(torch.zeros_like(outputs), 1 - targets_transformed * outputs)

    if class_weights is not None:
        weights_per_sample = torch.where(
            targets.view_as(outputs) == 1,
            class_weights[0],
            torch.tensor(1.0, device=targets.device),
        )
        loss = (loss_unreduced * weights_per_sample).mean()
    else:
        loss = loss_unreduced.mean()

    # Regularización L2
    l2_reg = sum(torch.sum(p ** 2) for name, p in model.named_parameters() if "weight" in name)
    return loss + weight_decay_lambda * l2_reg


def eval_model(model, loader, loss_fn, weight_decay_lambda=0.0):
    """Evalúa un modelo en un DataLoader.

    Args:
        model: Modelo PyTorch.
        loader: DataLoader con (X, y).
        loss_fn: Función de pérdida (hinge_loss_fn o BCEWithLogitsLoss).
        weight_decay_lambda: Para hinge loss; ignorado en BCE.

    Returns:
        (loss_promedio, accuracy)
    """
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            outputs = model(xb)
            if loss_fn == hinge_loss_fn:
                loss = loss_fn(outputs, yb.unsqueeze(1), weight_decay_lambda, model)
                preds = (outputs >= 0).float().squeeze()
            else:
                loss = loss_fn(outputs, yb.unsqueeze(1))
                preds = (torch.sigmoid(outputs) >= 0.5).float().squeeze()
            total_loss += loss.item() * xb.size(0)
            correct += (preds == yb).sum().item()
            total += xb.size(0)
    return total_loss / total, correct / total


def set_global_seed(seed: int):
    """Fija la semilla en todos los generadores para reproducibilidad."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)


def find_best_threshold(probs, y_true, metric_fn=f1_score):
    """Busca el umbral óptimo que maximiza una métrica en validación.

    Args:
        probs: Probabilidades predichas (array).
        y_true: Etiquetas reales (array).
        metric_fn: Función de métrica a maximizar (default: f1_score).

    Returns:
        Umbral óptimo (float).
    """
    best_score, best_thresh = 0.0, 0.5
    for thresh in np.arange(0.01, 1.0, 0.01):
        preds = (probs >= thresh).astype(int)
        score = metric_fn(y_true, preds, pos_label=1, zero_division=0)
        if score > best_score:
            best_score = score
            best_thresh = thresh
    return best_thresh


## 5. Carga de datos y particionado

In [ ]:
def load_data(path: str):
    """Carga y preprocesa el dataset de crisis epilépticas.

    Args:
        path: Ruta al CSV.

    Returns:
        (X_all, y_all) como tensores float32, ya escalados.
    """
    df = pd.read_csv(path, index_col=0)
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
    df["y_binary"] = (df["y"] == 1).astype(int)

    X = df.drop(["y", "y_binary"], axis=1).values
    y = df["y_binary"].values

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


def split_data(X_all, y_all, seed):
    """Divide en train/val/test (70/15/15).

    Args:
        X_all: Tensor de features.
        y_all: Tensor de etiquetas.
        seed: Semilla para reproducibilidad.

    Returns:
        dict con claves 'idx_train_val', 'idx_test',
        'X_train', 'X_val', 'X_test', 'y_train', 'y_val', 'y_test'.
    """
    idxs = np.arange(len(X_all))
    idx_train_val, idx_test = train_test_split(
        idxs, test_size=CONFIG["test_size"],
        random_state=seed, stratify=y_all.numpy(),
    )

    X_tv, y_tv = X_all[idx_train_val], y_all[idx_train_val]
    X_test, y_test = X_all[idx_test], y_all[idx_test]

    X_tr_np, X_val_np, y_tr_np, y_val_np = train_test_split(
        X_tv.numpy(), y_tv.numpy(),
        test_size=CONFIG["val_ratio"],
        random_state=seed, stratify=y_tv.numpy(),
    )

    return {
        "idx_train_val": idx_train_val,
        "idx_test": idx_test,
        "X_train": torch.tensor(X_tr_np, dtype=torch.float32),
        "X_val": torch.tensor(X_val_np, dtype=torch.float32),
        "X_test": X_test,
        "y_train": torch.tensor(y_tr_np, dtype=torch.float32),
        "y_val": torch.tensor(y_val_np, dtype=torch.float32),
        "y_test": y_test,
    }


def partition_iid(idx_train_val, num_clients, seed):
    """Partición IID: reparto aleatorio equitativo entre clientes.

    Args:
        idx_train_val: Índices del conjunto train+val.
        num_clients: Número de clientes federados.
        seed: Semilla.

    Returns:
        dict {client_id: [indices]}.
    """
    rnd = np.random.RandomState(seed)
    shuffled = rnd.permutation(idx_train_val)
    splits = np.array_split(shuffled, num_clients)
    return {i: splits[i].tolist() for i in range(num_clients)}


def partition_dirichlet(idx_train_val, y_all, num_clients, alpha, seed):
    """Partición No-IID mediante distribución de Dirichlet.

    Cada clase se reparte entre los clientes según proporciones
    muestreadas de Dir(alpha). Alpha bajo → más heterogeneidad.

    Args:
        idx_train_val: Índices del conjunto train+val.
        y_all: Etiquetas completas (numpy array).
        num_clients: Número de clientes federados.
        alpha: Parámetro de concentración de Dirichlet.
        seed: Semilla.

    Returns:
        dict {client_id: [indices_globales]}.
    """
    np.random.seed(seed)
    random.seed(seed)

    num_classes = len(np.unique(y_all))
    class_indices = [np.where(y_all == c)[0] for c in range(num_classes)]

    global_to_local = {g: l for l, g in enumerate(idx_train_val)}
    local_class_indices = [
        [global_to_local[g] for g in class_indices[c] if g in global_to_local]
        for c in range(num_classes)
    ]

    client_indices = {i: [] for i in range(num_clients)}

    for k in range(num_classes):
        proportions = np.random.dirichlet([alpha] * num_clients)
        np.random.shuffle(local_class_indices[k])

        current_idx = 0
        for cid in range(num_clients):
            n_samples = int(len(local_class_indices[k]) * proportions[cid])
            if cid == num_clients - 1:
                n_samples = len(local_class_indices[k]) - current_idx
            client_indices[cid].extend(local_class_indices[k][current_idx:current_idx + n_samples])
            current_idx += n_samples

    idx_list = idx_train_val.tolist()
    return {cid: [idx_list[l] for l in idxs] for cid, idxs in client_indices.items()}


## 6. Componentes federados (Flower)

In [ ]:
class SaveMetricsEarlyStopStrategy(FedAvg):
    """Estrategia FedAvg con early stopping basado en val_loss promedio.

    Registra métricas de entrenamiento/validación por ronda y detiene
    la simulación si no hay mejora durante `patience` rondas consecutivas.

    Args:
        delta: Mejora mínima requerida para considerar progreso.
        patience: Rondas sin mejora antes de detener.
        num_clients: Número mínimo de clientes por ronda.
    """

    def __init__(self, delta: float = 1e-4, patience: int = 20, num_clients: int = 10):
        super().__init__(
            fraction_fit=1.0,
            min_fit_clients=num_clients,
            min_available_clients=num_clients,
        )
        self.global_weights = None
        self.best_val_loss = float("inf")
        self.no_improve = 0
        self.delta = delta
        self.patience = patience

        self.train_losses = []
        self.val_losses = []
        self.train_accs = []
        self.val_accs = []

    def configure_fit(self, server_round, parameters, client_manager):
        if self.should_stop():
            return None
        sampled = client_manager.sample(
            num_clients=self.min_fit_clients,
            min_num_clients=self.min_fit_clients,
        )
        sampled_sorted = sorted(sampled, key=lambda c: int(c.cid))
        fit_ins = FitIns(parameters, {})
        return [(c, fit_ins) for c in sampled_sorted]

    def aggregate_fit(self, rnd, results, failures):
        params_agg, metrics_agg = super().aggregate_fit(rnd, results, failures)
        self.global_weights = params_agg

        self.train_losses.append(np.mean([r.metrics["train_loss"] for _, r in results]))
        self.val_losses.append(np.mean([r.metrics["val_loss"] for _, r in results]))
        self.train_accs.append(np.mean([r.metrics["train_accuracy"] for _, r in results]))
        self.val_accs.append(np.mean([r.metrics["val_accuracy"] for _, r in results]))

        current = self.val_losses[-1]
        if current < self.best_val_loss - self.delta:
            self.best_val_loss = current
            self.no_improve = 0
        else:
            self.no_improve += 1

        print(
            f"[Round {rnd}] "
            f"Train Loss={self.train_losses[-1]:.4f}, Val Loss={self.val_losses[-1]:.4f} | "
            f"Train Acc={self.train_accs[-1]:.4f}, Val Acc={self.val_accs[-1]:.4f} | "
            f"NoImprove={self.no_improve}/{self.patience}"
        )
        return params_agg, metrics_agg

    def aggregate_evaluate(self, rnd, results, failures):
        return 0.0, {}

    def should_stop(self) -> bool:
        return self.no_improve >= self.patience


def make_client_fn(model_type, client_indices, X_all, y_all, seed):
    """Genera la función client_fn para Flower simulation.

    Cada cliente recibe su subconjunto de datos, crea un modelo local,
    y entrena durante `local_epochs` epochs por ronda.

    Args:
        model_type: 'mlp' o 'svm'.
        client_indices: dict {client_id: [indices]}.
        X_all: Tensor completo de features.
        y_all: Tensor completo de etiquetas.
        seed: Semilla base (se suma client_id para cada cliente).

    Returns:
        Función client_fn(cid) compatible con fl.simulation.
    """
    batch_size = CONFIG[f"batch_size_{model_type}"]
    local_epochs = CONFIG["local_epochs"]

    def client_fn(cid: str):
        client_id = int(cid)
        local_seed = seed + client_id
        torch.manual_seed(local_seed)
        np.random.seed(local_seed)
        random.seed(local_seed)

        idx = client_indices[client_id]
        X_local = X_all[idx]
        y_local = y_all[idx]

        # Split local train/val
        if len(np.unique(y_local.numpy())) > 1:
            X_tr, X_vl, y_tr, y_vl = train_test_split(
                X_local.numpy(), y_local.numpy(),
                test_size=0.15, random_state=local_seed, stratify=y_local.numpy(),
            )
        else:
            X_tr, X_vl, y_tr, y_vl = train_test_split(
                X_local.numpy(), y_local.numpy(),
                test_size=0.15, random_state=local_seed, shuffle=True,
            )

        X_tr = torch.tensor(X_tr, dtype=torch.float32)
        X_vl = torch.tensor(X_vl, dtype=torch.float32)
        y_tr = torch.tensor(y_tr, dtype=torch.float32)
        y_vl = torch.tensor(y_vl, dtype=torch.float32)

        gen = torch.Generator().manual_seed(local_seed)
        train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True, generator=gen)
        val_loader = DataLoader(TensorDataset(X_vl, y_vl), batch_size=batch_size, shuffle=False,
                                generator=torch.Generator().manual_seed(local_seed))

        model = create_model(model_type, X_all.shape[1])

        # Configurar loss según modelo
        if model_type == "mlp":
            criterion = nn.BCEWithLogitsLoss()
            make_optimizer = lambda: optim.Adam(model.parameters(), lr=CONFIG["mlp_lr"])
        else:
            pos = (y_tr == 1).sum().item() or 1e-6
            neg = (y_tr == 0).sum().item() or 1e-6
            class_w = torch.tensor([neg / pos], dtype=torch.float32)
            wd = CONFIG["svm_weight_decay"]
            criterion = lambda out, tgt: hinge_loss_fn(out, tgt, wd, model, class_w)
            make_optimizer = lambda: optim.SGD(model.parameters(), lr=CONFIG["svm_lr"])

        class FlowerClient(fl.client.NumPyClient):
            """Cliente Flower para entrenamiento federado local."""

            def get_parameters(self, config=None):
                return [v.cpu().detach().numpy() for v in model.state_dict().values()]

            def set_parameters(self, parameters):
                sd = model.state_dict()
                for k, val in zip(sd.keys(), parameters):
                    sd[k] = torch.tensor(val)
                model.load_state_dict(sd)

            def fit(self, parameters, config):
                self.set_parameters(parameters)
                model.train()
                optimizer = make_optimizer()

                for _ in range(local_epochs):
                    for xb, yb in train_loader:
                        optimizer.zero_grad()
                        logits = model(xb)
                        loss = criterion(logits, yb.unsqueeze(1))
                        loss.backward()
                        optimizer.step()

                # Evaluar tras entrenamiento local
                model.eval()
                def _eval(loader):
                    tot_loss, corr, tot = 0.0, 0, 0
                    with torch.no_grad():
                        for xb, yb in loader:
                            logits = model(xb)
                            tot_loss += criterion(logits, yb.unsqueeze(1)).item() * xb.size(0)
                            if model_type == "mlp":
                                p = (torch.sigmoid(logits) >= 0.5).float().squeeze()
                            else:
                                p = (logits >= 0).float().squeeze()
                            corr += (p == yb).sum().item()
                            tot += xb.size(0)
                    return tot_loss / tot, corr / tot

                tl, ta = _eval(train_loader)
                vl, va = _eval(val_loader)
                return self.get_parameters(), len(X_tr), {
                    "train_loss": tl, "train_accuracy": ta,
                    "val_loss": vl, "val_accuracy": va,
                }

            def evaluate(self, parameters, config):
                return 0.0, len(X_vl), {"test_loss": 0.0, "test_accuracy": 0.0}

        return FlowerClient()

    return client_fn


## 7. Entrenamiento centralizado

In [ ]:
def train_centralized(model_type, model, train_loader, val_loader, y_train, seed):
    """Entrena un modelo de forma centralizada con early stopping.

    Args:
        model_type: 'mlp' o 'svm'.
        model: Modelo PyTorch.
        train_loader: DataLoader de entrenamiento.
        val_loader: DataLoader de validación.
        y_train: Tensor de etiquetas de entrenamiento (para calcular pesos de clase en SVM).
        seed: Semilla para reproducibilidad.

    Returns:
        dict con historial de métricas y número de epochs ejecutados.
    """
    max_epochs = CONFIG[f"{model_type}_max_epochs"]
    patience = CONFIG[f"{model_type}_patience_central"]

    if model_type == "mlp":
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=CONFIG["mlp_lr"])
    else:
        pos = (y_train == 1).sum().item() or 1e-6
        neg = (y_train == 0).sum().item() or 1e-6
        class_w = torch.tensor([neg / pos], dtype=torch.float32)
        wd = CONFIG["svm_weight_decay"]
        criterion = lambda out, tgt: hinge_loss_fn(out, tgt, wd, model, class_w)
        optimizer = optim.SGD(model.parameters(), lr=CONFIG["svm_lr"])
        print(f"  Peso clase positiva (Hinge): {class_w.item():.2f}")

    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    best_val_loss = float("inf")
    no_improve = 0
    best_state = None

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb.unsqueeze(1))
            loss.backward()
            optimizer.step()

        # Evaluar epoch
        if model_type == "mlp":
            tl, ta = eval_model(model, train_loader, nn.BCEWithLogitsLoss())
            vl, va = eval_model(model, val_loader, nn.BCEWithLogitsLoss())
        else:
            tl, ta = eval_model(model, train_loader, hinge_loss_fn)
            vl, va = eval_model(model, val_loader, hinge_loss_fn)

        train_losses.append(tl)
        val_losses.append(vl)
        train_accs.append(ta)
        val_accs.append(va)

        # Early stopping
        if vl + 1e-4 < best_val_loss:
            best_val_loss = vl
            no_improve = 0
            if model_type == "svm":
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1

        if no_improve >= patience:
            print(f"  Early stopping centralizado en epoch {epoch}")
            if model_type == "svm" and best_state:
                model.load_state_dict(best_state)
            break

    return {
        "train_loss": train_losses, "val_loss": val_losses,
        "train_acc": train_accs, "val_acc": val_accs,
        "epochs_executed": len(train_losses),
    }


## 8. Evaluación final en test

In [ ]:
def evaluate_on_test_mlp(model, test_loader):
    """Evalúa un modelo MLP en el conjunto de test.

    Usa sigmoid + umbral 0.5 para predicciones.

    Returns:
        dict con métricas (loss, accuracy, precision, recall, roc_auc),
        confusion_matrix, preds y probs.
    """
    model.eval()
    all_preds, all_true, all_probs = [], [], []

    with torch.no_grad():
        for xb, yb in test_loader:
            probs = torch.sigmoid(model(xb)).squeeze().cpu().numpy()
            preds = (probs >= 0.5).astype(int)
            all_probs.append(probs)
            all_preds.extend(preds.tolist())
            all_true.extend(yb.cpu().numpy().tolist())

    y_true = np.array(all_true)
    y_pred = np.array(all_preds)
    y_prob = np.concatenate(all_probs)

    return {
        "loss": log_loss(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "cm": confusion_matrix(y_true, y_pred),
        "y_true": y_true, "y_pred": y_pred, "y_prob": y_prob,
    }


def evaluate_on_test_svm(model, test_loader, val_X, val_y, seed):
    """Evalúa un modelo SVM calibrando probabilidades vía Platt scaling.

    Busca el umbral óptimo de F1 en validación y lo aplica al test.

    Returns:
        dict con métricas, confusion_matrix, best_threshold, preds y probs.
    """
    # Transferir pesos a sklearn SGDClassifier para calibración
    coef = model.linear.weight.detach().numpy().flatten()
    intercept = model.linear.bias.detach().numpy().flatten()

    dummy_clf = SklearnSGDClassifier(loss="hinge", random_state=seed, class_weight="balanced")
    dummy_clf.fit(np.zeros((2, coef.shape[0])), np.array([0, 1]))
    dummy_clf.coef_ = coef.reshape(1, -1)
    dummy_clf.intercept_ = intercept.reshape(1)

    calibrated = CalibratedClassifierCV(dummy_clf, method="sigmoid", cv="prefit")
    calibrated.fit(val_X.numpy(), val_y.numpy())

    # Test
    X_test_list, y_test_list = [], []
    for xb, yb in test_loader:
        X_test_list.append(xb.numpy())
        y_test_list.append(yb.numpy())
    X_test_np = np.concatenate(X_test_list)
    y_true = np.concatenate(y_test_list)

    y_prob = calibrated.predict_proba(X_test_np)[:, 1]

    # Umbral óptimo en validación
    val_probs = calibrated.predict_proba(val_X.numpy())[:, 1]
    best_thresh = find_best_threshold(val_probs, val_y.numpy())
    print(f"  Mejor umbral (F1 en val): {best_thresh:.2f}")

    y_pred = (y_prob >= best_thresh).astype(int)

    return {
        "loss": log_loss(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "cm": confusion_matrix(y_true, y_pred),
        "best_threshold": best_thresh,
        "y_true": y_true, "y_pred": y_pred, "y_prob": y_prob,
    }


## 9. Visualización

In [ ]:
def _text_color(value, cmap, vmin, vmax):
    """Determina color de texto (blanco/negro) según luminancia del fondo."""
    norm = plt.Normalize(vmin, vmax)
    r, g, b = cmap(norm(value))[:3]
    return "white" if (0.299 * r + 0.587 * g + 0.114 * b) < 0.5 else "black"


def plot_training_curves(history_c, history_f, epochs_per_round=5):
    """Grafica curvas de loss y accuracy: centralizado vs federado.

    Args:
        history_c: dict con train_loss, val_loss, train_acc, val_acc (centralizado).
        history_f: dict con las mismas claves (federado, por rondas).
        epochs_per_round: Epochs locales por ronda federada (para eje X equivalente).
    """
    epochs_c = np.arange(1, len(history_c["train_loss"]) + 1)
    rounds_f = np.arange(1, len(history_f["train_loss"]) + 1)
    epochs_f = rounds_f * epochs_per_round

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    for ax, metric_t, metric_v, ylabel in [
        (ax1, "train_loss", "val_loss", "Loss"),
        (ax2, "train_acc", "val_acc", "Accuracy"),
    ]:
        ax.plot(epochs_c, history_c[metric_t], color="deepskyblue", lw=2, label=f"C Train {ylabel}")
        ax.plot(epochs_c, history_c[metric_v], color="navy", lw=2, label=f"C Val {ylabel}")
        ax.plot(epochs_f, history_f[metric_t], color="darkorange", lw=2, label=f"F Train {ylabel}")
        ax.plot(epochs_f, history_f[metric_v], color="crimson", lw=2, label=f"F Val {ylabel}")
        ax.set_xlabel("Épocas", fontsize=12)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(f"Training vs Validation {ylabel}", fontsize=14)
        ax.grid(color="gray", linestyle="--", linewidth=0.5, alpha=0.7)
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()


def plot_confusion_matrices(cm_c, cm_f):
    """Grafica 4 matrices: centralizado, federado, diferencia absoluta y porcentual.

    Args:
        cm_c: Confusion matrix centralizado (2x2 numpy array).
        cm_f: Confusion matrix federado (2x2 numpy array).
    """
    cm_diff = cm_c - cm_f
    cm_pct = np.zeros_like(cm_diff, dtype=float)
    for i in range(2):
        for j in range(2):
            if cm_f[i, j] != 0:
                cm_pct[i, j] = ((cm_c[i, j] - cm_f[i, j]) / cm_f[i, j]) * 100
            else:
                cm_pct[i, j] = 0 if cm_c[i, j] == 0 else np.nan

    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    cmap_main = plt.get_cmap("Blues")
    cmap_diff = plt.get_cmap("RdBu_r")

    configs = [
        (cm_c, "Confusión — Centralizado", cmap_main, False),
        (cm_f, "Confusión — Federado", cmap_main, False),
        (cm_diff, "Diferencia Absoluta (C − F)", cmap_diff, True),
        (cm_pct, "Diferencia Porcentual (C − F)", cmap_diff, True),
    ]

    for idx, (matrix, title, cmap, is_diff) in enumerate(configs):
        ax = axes[idx]
        safe = np.nan_to_num(matrix)
        vmin = -np.max(np.abs(safe)) if is_diff else safe.min()
        vmax = np.max(np.abs(safe)) if is_diff else safe.max()

        sns.heatmap(matrix, annot=False, cmap=cmap,
                    xticklabels=["Clase 0", "Clase 1"],
                    yticklabels=["Clase 0", "Clase 1"],
                    square=True, cbar=False,
                    center=0 if is_diff else None,
                    vmin=vmin, vmax=vmax, ax=ax)

        for ri in range(2):
            for ci in range(2):
                val = matrix[ri, ci]
                if np.isnan(val):
                    txt, clr = "NaN", "grey"
                else:
                    clr = _text_color(val, cmap, vmin, vmax)
                    txt = f"{val:.1f}%" if idx == 3 else f"{int(round(val))}"
                ax.text(ci + 0.5, ri + 0.5, txt, ha="center", va="center", color=clr, fontsize=12)

        ax.set_title(title, fontsize=14)
        ax.set_xlabel("Predicted Class", fontsize=12)
        ax.set_ylabel("True Class", fontsize=12)
        ax.set_aspect("equal")

    plt.tight_layout()
    plt.show()


## 10. Función principal: `run_simulation()`

In [ ]:
def run_simulation(model_type: str, distribution: str):
    """Ejecuta N simulaciones comparando centralizado vs federado.

    Cada simulación:
    1. Fija semilla.
    2. Carga datos y los divide en train/val/test.
    3. Entrena modelo centralizado con early stopping.
    4. Particiona datos entre clientes (IID o Dirichlet).
    5. Ejecuta simulación federada con FedAvg + early stopping.
    6. Evalúa ambos modelos en test, genera gráficos y guarda JSON.

    Args:
        model_type: 'mlp' o 'svm'.
        distribution: 'iid' o 'non-iid'.
    """
    n_sims = CONFIG["num_simulations"]
    batch_size = CONFIG[f"batch_size_{model_type}"]
    patience_fed = CONFIG[f"{model_type}_patience_federated"]
    tag = f"{model_type}_{distribution}".replace("-", "")

    print(f"\n{'#' * 60}")
    print(f"# {model_type.upper()} — {distribution.upper()} — {n_sims} simulaciones")
    print(f"{'#' * 60}\n")

    for i in range(1, n_sims + 1):
        seed = CONFIG["initial_seed"] + (i - 1)
        print(f"\n{'=' * 20} Simulación {i} — seed={seed} {'=' * 20}\n")

        set_global_seed(seed)

        # ── 1. Datos ─────────────────────────────────────────
        X_all, y_all = load_data(CONFIG["data_path"])
        data = split_data(X_all, y_all, seed)
        print(f"  Train: {len(data['X_train'])}, Val: {len(data['X_val'])}, Test: {len(data['X_test'])}")

        gen_seed = torch.Generator().manual_seed(seed)
        train_loader = DataLoader(TensorDataset(data["X_train"], data["y_train"]),
                                  batch_size=batch_size, shuffle=True, generator=gen_seed)
        val_loader = DataLoader(TensorDataset(data["X_val"], data["y_val"]),
                                batch_size=batch_size, shuffle=False,
                                generator=torch.Generator().manual_seed(seed))
        test_loader = DataLoader(TensorDataset(data["X_test"], data["y_test"]),
                                 batch_size=batch_size, shuffle=False,
                                 generator=torch.Generator().manual_seed(seed))

        # ── 2. Centralizado ──────────────────────────────────
        print("\n  ── Entrenamiento centralizado ──")
        model_c = create_model(model_type, X_all.shape[1])
        hist_c = train_centralized(model_type, model_c, train_loader, val_loader, data["y_train"], seed)
        print(f"  Epochs ejecutados: {hist_c['epochs_executed']}")

        # Evaluación centralizado
        if model_type == "mlp":
            res_c = evaluate_on_test_mlp(model_c, test_loader)
        else:
            res_c = evaluate_on_test_svm(model_c, test_loader, data["X_val"], data["y_val"], seed)

        print(f"\n  --- Métricas CENTRALIZADAS (Sim {i}) ---")
        for k in ["loss", "accuracy", "precision", "recall", "roc_auc"]:
            print(f"    {k}: {res_c[k]:.4f}")
        print(f"\n  Classification Report:\n{classification_report(res_c['y_true'], res_c['y_pred'], target_names=['Clase 0', 'Clase 1'], zero_division=0)}")

        # ── 3. Particionado ──────────────────────────────────
        if distribution == "iid":
            client_idx = partition_iid(data["idx_train_val"], CONFIG["num_clients"], seed)
        else:
            client_idx = partition_dirichlet(
                data["idx_train_val"], y_all.numpy(),
                CONFIG["num_clients"], CONFIG["dirichlet_alpha"], seed,
            )

        # ── 4. Federado ──────────────────────────────────────
        print("\n  ── Simulación federada ──")
        strategy = SaveMetricsEarlyStopStrategy(
            delta=1e-4, patience=patience_fed, num_clients=CONFIG["num_clients"],
        )
        client_fn = make_client_fn(model_type, client_idx, X_all, y_all, seed)

        fl.simulation.start_simulation(
            client_fn=client_fn,
            num_clients=CONFIG["num_clients"],
            strategy=strategy,
            config=ServerConfig(num_rounds=CONFIG["num_rounds"]),
        )
        print(f"  Apagando Ray...")
        ray.shutdown()
        time.sleep(5)

        num_rounds = len(strategy.val_losses)
        print(f"  Rondas ejecutadas: {num_rounds}")

        # Reconstruir modelo federado global
        model_f = create_model(model_type, X_all.shape[1])
        global_arrays = parameters_to_ndarrays(strategy.global_weights)
        sd = model_f.state_dict()
        for k, arr in zip(sd.keys(), global_arrays):
            sd[k] = torch.tensor(arr)
        model_f.load_state_dict(sd)
        model_f.eval()

        # Evaluación federado
        if model_type == "mlp":
            res_f = evaluate_on_test_mlp(model_f, test_loader)
        else:
            res_f = evaluate_on_test_svm(model_f, test_loader, data["X_val"], data["y_val"], seed)

        print(f"\n  --- Métricas FEDERADAS (Sim {i}) ---")
        for k in ["loss", "accuracy", "precision", "recall", "roc_auc"]:
            print(f"    {k}: {res_f[k]:.4f}")
        print(f"\n  Classification Report:\n{classification_report(res_f['y_true'], res_f['y_pred'], target_names=['Clase 0', 'Clase 1'], zero_division=0)}")

        # ── 5. Visualización ─────────────────────────────────
        hist_f = {
            "train_loss": strategy.train_losses, "val_loss": strategy.val_losses,
            "train_acc": strategy.train_accs, "val_acc": strategy.val_accs,
        }
        plot_training_curves(hist_c, hist_f, epochs_per_round=CONFIG["local_epochs"])
        plot_confusion_matrices(res_c["cm"], res_f["cm"])

        # ── 6. Guardar JSON ──────────────────────────────────
        results = {
            "seed": seed,
            "model_type": model_type,
            "distribution": distribution,
            "centralized": {
                "test_metrics": {k: res_c[k] for k in ["loss", "accuracy", "precision", "recall", "roc_auc"]},
                "confusion_matrix": res_c["cm"].tolist(),
                "history": {
                    "train_loss": hist_c["train_loss"], "val_loss": hist_c["val_loss"],
                    "train_accuracy": hist_c["train_acc"], "val_accuracy": hist_c["val_acc"],
                },
                "epochs_executed": hist_c["epochs_executed"],
            },
            "federated": {
                "test_metrics": {k: res_f[k] for k in ["loss", "accuracy", "precision", "recall", "roc_auc"]},
                "confusion_matrix": res_f["cm"].tolist(),
                "history": {
                    "train_loss": strategy.train_losses, "val_loss": strategy.val_losses,
                    "train_accuracy": strategy.train_accs, "val_accuracy": strategy.val_accs,
                },
                "rounds_executed": num_rounds,
            },
        }

        fname = f"simulation_results_{tag}_seed_{seed}.json"
        with open(fname, "w") as f:
            json.dump(results, f, indent=4)
        print(f"\n  Resultados guardados en '{fname}'")
        print(f"\n{'=' * 20} Simulación {i} finalizada {'=' * 20}")


## 11. Ejecución: MLP — IID

In [ ]:
run_simulation(model_type="mlp", distribution="iid")

## 12. Ejecución: MLP — Non-IID

In [ ]:
run_simulation(model_type="mlp", distribution="non-iid")

## 13. Ejecución: SVM — IID

In [ ]:
run_simulation(model_type="svm", distribution="iid")

## 14. Ejecución: SVM — Non-IID

In [ ]:
run_simulation(model_type="svm", distribution="non-iid")

## 15. Agregación de resultados

In [ ]:
def aggregate_results(model_type: str, distribution: str, epochs_per_round: int = 5):
    """Carga los JSON de N simulaciones y calcula métricas promedio.

    Genera gráficos de curvas promedio y matrices de confusión promedio.

    Args:
        model_type: 'mlp' o 'svm'.
        distribution: 'iid' o 'non-iid'.
        epochs_per_round: Epochs locales por ronda federada.
    """
    tag = f"{model_type}_{distribution}".replace("-", "")
    n = CONFIG["num_simulations"]

    c_metrics, f_metrics = [], []
    c_histories = {"train_loss": [], "val_loss": [], "train_accuracy": [], "val_accuracy": []}
    f_histories = {"train_loss": [], "val_loss": [], "train_accuracy": [], "val_accuracy": []}
    c_cms, f_cms = [], []

    for i in range(n):
        seed = CONFIG["initial_seed"] + i
        fname = f"simulation_results_{tag}_seed_{seed}.json"
        if not os.path.exists(fname):
            print(f"  Archivo no encontrado: {fname}")
            continue
        with open(fname) as f:
            data = json.load(f)

        if data.get("centralized", {}).get("test_metrics"):
            c_metrics.append(data["centralized"]["test_metrics"])
        if data.get("federated", {}).get("test_metrics"):
            f_metrics.append(data["federated"]["test_metrics"])

        for key in c_histories:
            h = data.get("centralized", {}).get("history", {}).get(key, [])
            if h:
                c_histories[key].append(h)
            h = data.get("federated", {}).get("history", {}).get(key, [])
            if h:
                f_histories[key].append(h)

        cm = data.get("centralized", {}).get("confusion_matrix")
        if cm:
            c_cms.append(np.array(cm))
        cm = data.get("federated", {}).get("confusion_matrix")
        if cm:
            f_cms.append(np.array(cm))

    # ── Métricas promedio ─────────────────────────────────
    print(f"\n{'#' * 50}")
    print(f"# AGREGACIÓN: {model_type.upper()} — {distribution.upper()}")
    print(f"{'#' * 50}")

    if c_metrics:
        print("\n  Promedio Centralizado:")
        for k in c_metrics[0]:
            avg = np.mean([m[k] for m in c_metrics])
            print(f"    {k}: {avg:.4f}")

    if f_metrics:
        print("\n  Promedio Federado:")
        for k in f_metrics[0]:
            avg = np.mean([m[k] for m in f_metrics])
            print(f"    {k}: {avg:.4f}")

    # ── Curvas promedio ───────────────────────────────────
    def avg_history(hist_list):
        if not hist_list:
            return []
        max_len = max(len(h) for h in hist_list)
        means = []
        for t in range(max_len):
            vals = [h[t] for h in hist_list if len(h) > t]
            if len(vals) >= 1:
                means.append(float(np.mean(vals)))
            else:
                break
        return means

    avg_c = {k: avg_history(v) for k, v in c_histories.items()}
    avg_f = {k: avg_history(v) for k, v in f_histories.items()}

    # Renombrar keys para compatibilidad con plot_training_curves
    hist_c_plot = {
        "train_loss": avg_c["train_loss"], "val_loss": avg_c["val_loss"],
        "train_acc": avg_c["train_accuracy"], "val_acc": avg_c["val_accuracy"],
    }
    hist_f_plot = {
        "train_loss": avg_f["train_loss"], "val_loss": avg_f["val_loss"],
        "train_acc": avg_f["train_accuracy"], "val_acc": avg_f["val_accuracy"],
    }

    if hist_c_plot["train_loss"] and hist_f_plot["train_loss"]:
        plot_training_curves(hist_c_plot, hist_f_plot, epochs_per_round)

    # ── Confusion matrices promedio ───────────────────────
    avg_cm_c = np.mean(c_cms, axis=0) if c_cms else np.zeros((2, 2))
    avg_cm_f = np.mean(f_cms, axis=0) if f_cms else np.zeros((2, 2))

    if np.any(avg_cm_c != 0) or np.any(avg_cm_f != 0):
        plot_confusion_matrices(avg_cm_c, avg_cm_f)


### Agregar resultados MLP

In [ ]:
aggregate_results("mlp", "iid")
aggregate_results("mlp", "non-iid")


### Agregar resultados SVM

In [ ]:
aggregate_results("svm", "iid")
aggregate_results("svm", "non-iid")
